# 02 – SQL Sanity Checks
The focus is to ensure:
- Primary keys are unique
- Foreign key relationships are valid
- No unexpected NULLs exist in critical columns
- Table cardinalities match business expectations

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("instacart.db")


## Primary Key Validation

- Each order should have a unique order_id.
- Each product should have a unique 'product_id'
- Each Aisle should have a unique 'Aisle_id'
- Each department should have a unique 'department_id'
Duplicates would indicate data corruption or ingestion errors.


In [2]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM orders;""",conn)


,total_rows,unique_orders
0,3421083,3421083


In [4]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS unique_products
FROM products;""",conn)


,total_rows,unique_products
0,49688,49688


In [6]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Aisle_id) AS unique_Aisle
FROM Aisles;""",conn)


,total_rows,unique_Aisle
0,134,134


In [7]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT department_id) AS unique_department
FROM departments;""",conn)


,total_rows,unique_department
0,21,21


In [8]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id || '-' || product_id) AS unique_pairs
FROM order_products_prior;""",conn)


,total_rows,unique_pairs
0,32434489,32434489


In [9]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id || '-' || product_id) AS unique_pairs
FROM order_products_train;""",conn)


,total_rows,unique_pairs
0,1384617,1384617


## User–Order Relationship Validation

Each order must belong to exactly one user.
This check ensures:
- `user_id` is never NULL
- Every order is associated with a user


In [10]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_ids
FROM orders;
""", conn)


,total_orders,null_user_ids
0,3421083,0


## Foreign Key Validation

In [11]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM order_products_prior opp
LEFT JOIN orders o ON opp.order_id = o.order_id
WHERE o.order_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


In [12]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM order_products_train opt
LEFT JOIN orders o ON opt.order_id = o.order_id
WHERE o.order_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


In [13]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM order_products_prior opp
LEFT JOIN products o ON opp.product_id = o.product_id
WHERE o.product_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


In [14]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM order_products_train opt
LEFT JOIN products o ON opt.product_id = o.product_id
WHERE o.product_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


In [15]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM products p
LEFT JOIN aisles a ON p.aisle_id = a.aisle_id
WHERE a.aisle_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


In [16]:
query = """
SELECT COUNT(*) AS orphan_rows
FROM products p
LEFT JOIN departments d ON p.department_id = d.department_id
WHERE d.department_id IS NULL;
"""
pd.read_sql(query,conn)

,orphan_rows
0,0


## Validate Reorder Flag Values

The `reordered` column should only contain binary values (0 or 1).

In [17]:
pd.read_sql("""
SELECT DISTINCT reordered
FROM order_products_prior;
""", conn)


,reordered
0,1
1,0


In [18]:
pd.read_sql("""
SELECT DISTINCT reordered
FROM order_products_train;
""", conn)


,reordered
0,1
1,0


## Validate add_to_cart_order Values

The add-to-cart position should:
- Be non-null
- Start from 1
- Be positive integers


In [19]:
pd.read_sql("""
SELECT
    MIN(add_to_cart_order) AS min_position,
    MAX(add_to_cart_order) AS max_position
FROM order_products_prior;
""", conn)



,min_position,max_position
0,1,145


In [20]:
pd.read_sql("""
SELECT
    MIN(add_to_cart_order) AS min_position,
    MAX(add_to_cart_order) AS max_position
FROM order_products_train;
""", conn)


,min_position,max_position
0,1,80


## Order Sequence Sanity Check

Each user's orders should follow a logical sequence
with increasing `order_number`.


In [21]:
pd.read_sql("""
SELECT user_id
FROM orders
GROUP BY user_id
HAVING MAX(order_number) != COUNT(order_id)
LIMIT 10;
""", conn)


,user_id


## Summary

Key observations:
- Primary keys are unique
- Foreign key relationships are intact
- No invalid or unexpected values detected
- Dataset is structurally reliable for analysis